In [1]:
import pandas as pd

buoy, locs = 'ARTG', ['btm1','btm2','sfc']
# buoy, locs = 'CLIS', ['btm','sfc']
# buoy, locs = 'EXRX', ['btm1','btm2','mid','sfc']
# buoy, locs = 'WLIS', ['btm1','btm2','mid','sfc']

# Variables and ranges
vars = ['T', 'S', 'DO', 'P', 'C', 'pH', 'rho', 'DOsat']
ranges = {'T': (-5, 30), 'S': (5, 35), 'DO': (0, 20), 'P': (0, 50),
          'C': (0, 50),'pH': (6, 10), 'rho': (16, 25), 'DOsat': (0, 150)}

for loc in locs:
    # Load dataset
    df = pd.read_csv(f'{buoy}_{loc}_QAQC.csv')

    for var in vars:
        d_col, q_col, c_col = f'{var}_data', f'{var}_Q', f'{var}_FailedCount'
        
        # Modify QA/QC flags
        q = df[q_col].astype(str).str.zfill(5)
        first, rest = q.str[0], q.str[1:]
        first = first.replace({'3':'1', '4':'3'})
        mask = df[d_col].isna() | (df[d_col] < ranges[var][0]) | (df[d_col] > ranges[var][1])
        first.loc[mask] = '4'
        
        # Update table
        new_q = first + rest
        df[q_col] = new_q.astype(int)
        df[c_col] = new_q.str.count('4').astype(int)
    
    df.to_csv(f'{buoy}_{loc}_QAQC_new.csv', index=False)

In [2]:
import pandas as pd

flags = {'Pass': 1, 'Suspect': 3, 'Fail': 4}
results = []

for stn in ['A4', 'C1', 'E1', 'I2']:
    # Load dataset
    df = pd.read_csv(f'DEEP_{stn}_WQ_QAQC.csv')
    df['time'] = pd.to_datetime(df['time'], errors='coerce')
    df = df[df['time'] < '2025-01-01']
    
    # Statistics of QA/QC flags
    for var in ['T', 'S', 'DO']:
        row = {'Station': stn, 'Variable': var}
        for label, flag in flags.items():
            count = (df[f'{var}_Q'] == flag).sum()
            pct = 100 * count / len(df)
            row[label] = f'{count} ({pct:.1f}%)'
        results.append(row)

qaqc_summary = pd.DataFrame(results)
qaqc_summary

,Station,Variable,Pass,Suspect,Fail
0,A4,T,84945 (93.6%),2484 (2.7%),3322 (3.7%)
1,A4,S,81491 (89.8%),4455 (4.9%),4805 (5.3%)
2,A4,DO,81933 (90.3%),3792 (4.2%),5026 (5.5%)
3,C1,T,49493 (92.2%),857 (1.6%),3304 (6.2%)
4,C1,S,49154 (91.6%),893 (1.7%),3607 (6.7%)
5,C1,DO,48754 (90.9%),653 (1.2%),4247 (7.9%)
6,E1,T,89589 (93.3%),2034 (2.1%),4416 (4.6%)
7,E1,S,88276 (91.9%),3116 (3.2%),4647 (4.8%)
8,E1,DO,89541 (93.2%),982 (1.0%),5516 (5.7%)
9,I2,T,61396 (91.0%),1873 (2.8%),4168 (6.2%)


In [3]:
buoy, locs = 'ARTG', ['btm1','btm2','sfc']
qaqc_tests = ['Threshold', 'JumpLim', 'Gap', 'PresRng', 'Spike']
results = []

for loc in locs:
    # Load dataset
    df = pd.read_csv(f'{buoy}_{loc}_QAQC.csv')
    df['TmStamp'] = pd.to_datetime(df['TmStamp'], format='%d-%b-%Y %H:%M:%S', errors='coerce')
    df = df[df['TmStamp'] < '2025-01-01']

    # Statistics of QA/QC tests
    for var in ['T', 'S', 'DO']:
        q = df[f'{var}_Q'].astype(str).str.zfill(5)
        row = {'Location': loc, 'Variable': var}
        for i, test in enumerate(qaqc_tests):
            fail_count = (q.str[i].astype(int) == 4).sum()
            fail_pct = 100 * fail_count / len(q)
            row[test] = f'{fail_count} ({fail_pct:.1f}%)'
        results.append(row)

qaqc_summary = pd.DataFrame(results)
qaqc_summary

,Location,Variable,Threshold,JumpLim,Gap,PresRng,Spike
0,btm1,T,279 (0.1%),487 (0.2%),59 (0.0%),290 (0.1%),487 (0.2%)
1,btm1,S,279 (0.1%),487 (0.2%),59 (0.0%),290 (0.1%),487 (0.2%)
2,btm1,DO,287 (0.1%),487 (0.2%),59 (0.0%),290 (0.1%),487 (0.2%)
3,btm2,T,282 (0.1%),476 (0.2%),909 (0.5%),289 (0.1%),476 (0.2%)
4,btm2,S,283 (0.1%),476 (0.2%),909 (0.5%),289 (0.1%),476 (0.2%)
5,btm2,DO,2801 (1.4%),2986 (1.5%),909 (0.5%),289 (0.1%),2987 (1.5%)
6,sfc,T,295 (0.1%),502 (0.2%),26 (0.0%),312 (0.2%),502 (0.2%)
7,sfc,S,5811 (2.8%),502 (0.2%),26 (0.0%),312 (0.2%),502 (0.2%)
8,sfc,DO,325 (0.2%),502 (0.2%),26 (0.0%),312 (0.2%),502 (0.2%)
